# Verified Reasoning Monitoring: development smoke

This notebook provisions the pinned Linux verifier first, then runs only the registered 32-task by four-candidate development feasibility check. It does not evaluate H1-H3. A failed setup is an operational result, not a scientific feasibility failure.

In [ ]:
from google.colab import userdata
from pathlib import Path
from urllib.request import urlopen, urlretrieve
import hashlib
import json
import os
import re
import shutil
import subprocess
import tarfile

PROJECT_GIT_URL = ''  # Set to the repository HTTPS URL after an immutable commit exists.
PROJECT_GIT_REV = ''  # Exact 40-character commit; branches and tags are rejected.
PROJECT = Path('/content/verified-reasoning-monitoring')
RUNTIME = Path('/content/vrm-runtime')
TOOLS = RUNTIME / 'tools'
CACHE = RUNTIME / 'mathlib4-v3'
BENCHMARK_RELEASE = RUNTIME / 'benchmark-release'
ARCHIVE = RUNTIME / 'leandojo-benchmark-4-v3.tar.gz'
PREPARED = Path('/content/vrm-artifacts/prepared-v2')
RUN = Path('/content/vrm-artifacts/development-smoke-v1')

assert re.fullmatch(r'^[0-9a-f]{40}$', PROJECT_GIT_REV), 'Use an exact project commit.'
assert PROJECT_GIT_URL.startswith('https://github.com/'), 'Use the reviewable HTTPS repository.'
hf_token = userdata.get("HF_TOKEN")
assert hf_token, 'Add HF_TOKEN through the Colab Secrets panel.'
os.environ['HF_TOKEN'] = hf_token

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', PROJECT_GIT_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', PROJECT_GIT_REV], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', PROJECT_GIT_REV], check=True)
head = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
assert head == PROJECT_GIT_REV
os.chdir(PROJECT)


## 1. Provision pinned verifier dependencies

This cell does not load Gemma. It installs an exact Go release, the frozen Lean toolchain, pinned Python dependencies, Landrun, Comparator, and the official LeanDojo cache. Every source checkout is verified at its immutable commit.

In [ ]:
# provision-verifier
def run(command, *, cwd=None, env=None):
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True)

def exact_checkout(url, revision, destination):
    if not destination.exists():
        run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)])
    run(['git', '-C', str(destination), 'fetch', 'origin', revision])
    run(['git', '-C', str(destination), 'checkout', '--detach', revision])
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == revision

RUNTIME.mkdir(parents=True, exist_ok=True)
TOOLS.mkdir(parents=True, exist_ok=True)
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'curl', 'elan', 'git', 'util-linux'])

go_version = 'go1.24.0'
go_metadata = json.load(urlopen('https://go.dev/dl/?mode=json&include=all'))
go_file = next(
    file
    for release in go_metadata if release['version'] == go_version
    for file in release['files']
    if file['os'] == 'linux' and file['arch'] == 'amd64' and file['kind'] == 'archive'
)
go_archive = RUNTIME / go_file['filename']
if not go_archive.exists():
    urlretrieve('https://go.dev/dl/' + go_file['filename'], go_archive)
assert hashlib.sha256(go_archive.read_bytes()).hexdigest() == go_file['sha256']
go_root = TOOLS / go_version
if not go_root.exists():
    go_root.mkdir()
    with tarfile.open(go_archive) as archive_handle:
        archive_handle.extractall(go_root, filter='data')
go_bin = go_root / 'go' / 'bin'

elan_home = TOOLS / 'elan'
env = dict(os.environ, ELAN_HOME=str(elan_home))
env['PATH'] = f"{elan_home / 'bin'}:{go_bin}:{TOOLS / 'bin'}:{env['PATH']}"
run(['elan', 'toolchain', 'install', 'leanprover/lean4:v4.29.0-rc2'], env=env)
run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], env=env)
run(['python', '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], env=env)

landrun_source = TOOLS / 'landrun-source'
exact_checkout('https://github.com/Zouuup/landrun', '811cfff51ceaf3d9843708aa6d22e9b84ccac8b4', landrun_source)
(TOOLS / 'bin').mkdir(exist_ok=True)
landrun_binary = TOOLS / 'bin' / 'landrun'
run([str(go_bin / 'go'), 'build', '-trimpath', '-o', str(landrun_binary), './cmd/landrun'], cwd=landrun_source, env=env)

comparator = TOOLS / 'comparator'
exact_checkout('https://github.com/leanprover/comparator', '3090445149fbaba51d8177df4eb2121573788341', comparator)
run(['lake', 'build', 'lean4export', 'comparator'], cwd=comparator, env=env)

from lean_dojo import LeanGitRepo, get_traced_repo_path, is_available_in_cache
repo = LeanGitRepo('https://github.com/leanprover-community/mathlib4', '1bc7728a050fc18ca2683f614c531cd7050ff063')
assert is_available_in_cache(repo), 'Official pretraced cache unavailable; do not trace Mathlib in this study.'
if not CACHE.exists():
    shutil.copytree(get_traced_repo_path(repo), CACHE, symlinks=True)

env['PATH'] = f"{TOOLS / 'bin'}:{env['PATH']}"
os.environ.update(env)
from vrm.lean import cache_digest
CACHE_SHA256 = cache_digest(CACHE)
LANDRUN_SHA256 = hashlib.sha256(landrun_binary.read_bytes()).hexdigest()
provision_receipt = {
    'project_commit': PROJECT_GIT_REV,
    'cache_sha256': CACHE_SHA256,
    'landrun_sha256': LANDRUN_SHA256,
    'comparator_commit': '3090445149fbaba51d8177df4eb2121573788341',
    'lean_version': 'v4.29.0-rc2',
    'go_version': go_version,
}
Path('/content/vrm-artifacts').mkdir(parents=True, exist_ok=True)
Path('/content/vrm-artifacts/provision-receipt.json').write_text(json.dumps(provision_receipt, sort_keys=True))
provision_receipt


## 2. Run the trusted-verifier preflight

Colab normally starts as root, so security-sensitive commands run as a dedicated unprivileged account. This must return `ready`; otherwise stop before loading Gemma.

In [ ]:
subprocess.run(['id', 'vrmrunner'], capture_output=True).returncode == 0 or run(['useradd', '--create-home', 'vrmrunner'])
run(['mkdir', '-p', '/content/vrm-artifacts'])
run(['chown', '-R', 'vrmrunner:vrmrunner', '/content/vrm-artifacts'])

study_env = dict(os.environ)
study_env.update({
    'HOME': '/home/vrmrunner',
    'VRM_LEAN_EXECUTION_MODE': 'native',
    'VRM_LEAN_CACHE': str(CACHE),
    'VRM_LEAN_CACHE_SHA256': CACHE_SHA256,
    'VRM_LANDRUN_SHA256': LANDRUN_SHA256,
    'VRM_COMPARATOR_ROOT': str(comparator),
})
def as_study_user(command):
    return subprocess.run(
        ['runuser', '--preserve-environment', '-u', 'vrmrunner', '--', *command],
        env=study_env, text=True, capture_output=True,
    )

preflight = as_study_user([
    'vrm', 'preflight', '--execution-mode', 'native',
    '--cache-dir', str(CACHE), '--cache-sha256', CACHE_SHA256,
    '--landrun-sha256', LANDRUN_SHA256, '--comparator-root', str(comparator),
])
print(preflight.stdout)
assert preflight.returncode == 0, preflight.stderr


## 3. Fetch and verify the official benchmark archive

The Zenodo API is queried by immutable record ID. The file is selected by the frozen MD5, not by a mutable filename.

In [ ]:
archive_md5 = 'b58af89599d5bbc792abc3744e5d37d9'
if not ARCHIVE.exists():
    record = json.load(urlopen('https://zenodo.org/api/records/18815372'))
    source_file = next(item for item in record['files'] if item['checksum'] == 'md5:' + archive_md5)
    urlretrieve(source_file['links']['self'], ARCHIVE)
assert hashlib.md5(ARCHIVE.read_bytes(), usedforsecurity=False).hexdigest() == archive_md5
if not BENCHMARK_RELEASE.exists():
    BENCHMARK_RELEASE.mkdir()
    with tarfile.open(ARCHIVE) as archive_handle:
        archive_handle.extractall(BENCHMARK_RELEASE, filter='data')
benchmark_candidates = list(BENCHMARK_RELEASE.rglob('novel_premises/train.json'))
assert len(benchmark_candidates) == 1
BENCHMARK = benchmark_candidates[0].parents[1]
BENCHMARK


## 4. Prepare the frozen development population

Preparation is deterministic and refuses to overwrite an existing directory. Reuse an existing output only after validating its manifest.

In [ ]:
if not PREPARED.exists():
    prepared = as_study_user([
        'vrm', 'prepare', '--config', str(PROJECT / 'configs/study.json'),
        '--benchmark-root', str(BENCHMARK), '--source-root', str(CACHE),
        '--archive', str(ARCHIVE), '--output', str(PREPARED),
    ])
    print(prepared.stdout)
    assert prepared.returncode == 0, prepared.stderr
manifest = json.loads(PREPARED.joinpath('manifest.json').read_text())
assert manifest['status'] == 'ready' and manifest['counts']['dev'] == 32
manifest['counts']


## 5. Run or resume the registered 32 x 4 smoke

Only now is Gemma loaded. The command resumes matching immutable candidate artifacts and retains infrastructure failures with their measured cost.

In [ ]:
smoke = as_study_user([
    'vrm', 'smoke', '--config', str(PROJECT / 'configs/study.json'),
    '--prepared-dir', str(PREPARED), '--output', str(RUN),
    '--execution-mode', 'native', '--cache-dir', str(CACHE),
    '--cache-sha256', CACHE_SHA256, '--landrun-sha256', LANDRUN_SHA256,
    '--comparator-root', str(comparator),
])
print(smoke.stdout)
if smoke.stderr:
    print(smoke.stderr)
assert RUN.joinpath('summary.json').is_file(), 'No scientific summary was produced.'
summary = json.loads(RUN.joinpath('summary.json').read_text())
summary


## 6. Apply the mechanical gate

Continue to protected monitor work only when all 128 candidates completed and the frozen valid, invalid, and mixed-task thresholds passed. Operational failures leave H1-H3 untested.

In [ ]:
decision = {
    'run_status': summary['status'],
    'completed_candidates': summary['completed_candidates'],
    'failed_attempts': summary['failed_attempts'],
    'feasibility': summary['feasibility'],
    'continue_to_H1_H3': bool(
        summary['status'] == 'completed'
        and summary['completed_candidates'] == 128
        and summary['feasibility']
        and summary['feasibility']['passed']
    ),
}
decision
